In [69]:
print("Hi")

Hi


In [70]:
neshan_weight_path = "data/neshan-sample-weights.json"
init_seed_weight_path = "data/init-seed-weights.json"

In [71]:
from sumolib import net
import json
import random

net_obj = net.readNet("map.net.xml")

neshan_weights = {}
init_seed_weights = {}

for edge in net_obj.getEdges():
    if edge.getID().startswith(":"):
        continue  # skip internal edges

    neshan_weights[edge.getID()] = edge.getLength()  # or speed, lanes, etc.
    init_seed_weights[edge.getID()] = random.randint(1, 100)  # random weight for seed initialization
# print(neshan_weight)
# print(init_seed_weights)

with open(neshan_weight_path, "w") as f:
    json.dump(neshan_weights, f, indent=2)
    
with open(init_seed_weight_path, "w") as f:
    json.dump(init_seed_weights, f, indent=2)

In [72]:
import json

with open(neshan_weight_path, "r") as f:
    neshan_weights = json.load(f)

with open(init_seed_weight_path, "r") as f:
    init_seed_weights = json.load(f)
    
print("Neshan Weights:", neshan_weights)
print("Initial Seed Weights:", init_seed_weights)
    

Neshan Weights: {'-1035578872': 211.88, '-1035578874': 16.13, '-1035578887#0': 46.95, '-1035578887#1': 80.64, '-1035578887#2': 118.61, '-1199956439': 4.57, '-1385521080': 119.91, '-1461800851': 0.2, '-164595013': 84.23, '-227547757#0': 0.2, '-227547757#1': 31.92, '-227547757#2': 45.25, '-313324028': 54.63, '-313324686#0': 113.39, '-313324686#1': 159.31, '-313324688': 123.08, '-39855493#0': 127.27, '-39855493#1': 112.99, '-39855493#2': 120.22, '-39855493#3': 120.08, '-39855493#4': 121.43, '-39855493#5': 117.75, '-42463779#0': 146.77, '-42463779#1': 46.8, '-42463779#2': 47.36, '-42463779#3': 0.2, '-4337771#0': 40.65, '-4337771#1': 51.36, '-4337771#2': 50.7, '-4337771#3': 47.27, '-4337773': 0.2, '-443790550': 182.6, '-443790552': 77.32, '-443790553': 46.8, '-445319881': 27.54, '-445321251#0': 121.96, '-445321251#1': 20.95, '-445321251#2': 17.21, '-482796570#0': 199.1, '-482796570#1': 66.9, '-482796570#2': 117.12, '-482796570#3': 120.87, '-482796570#4': 120.71, '-482796570#5': 124.45, '-48

In [73]:
import xml.etree.ElementTree as ET

tree = ET.parse("edges.xml")
root = tree.getroot()

edges = []

for interval in root.findall("interval"):
    begin = float(interval.attrib["begin"])
    end = float(interval.attrib["end"])

    for edge in interval.findall("edge"):
        edges.append({
            "id": edge.attrib["id"],
            "traveltime": float(edge.attrib.get("traveltime", 0)),
            "interval_begin": begin,
            "interval_end": end,
            "speed": float(edge.attrib.get("speed", 0)),
            "density": float(edge.attrib.get("density", 0)),
            "timeLoss": float(edge.attrib.get("timeLoss", 0)),
            "waitingTime": float(edge.attrib.get("waitingTime", 0)),
        })

print(f"Loaded {len(edges)} edge records")
print(edges[0])

Loaded 145 edge records
{'id': '-1035578887#0', 'traveltime': 7.21, 'interval_begin': 0.0, 'interval_end': 3571.0, 'speed': 6.33, 'density': 0.04, 'timeLoss': 1.28, 'waitingTime': 0.0}


In [74]:
!python "C:\Program Files (x86)\Eclipse\Sumo\tools\district\gridDistricts.py" -n map.net.xml -o grids.taz.xml -w 1000

In [75]:
import xml.etree.ElementTree as ET
import numpy as np

tree = ET.parse("grids.taz.xml")
root = tree.getroot()

# extract all zone IDs
zones = [t.attrib["id"] for t in root.findall("taz")]
n = len(zones)

zone_index = {z: i for i, z in enumerate(zones)}

print("Zones:", n)

init_od_matrix = np.zeros((n, n), dtype=int)

import random
for i in range(n):
    for j in range(n):
        if i == j:
            continue
        init_od_matrix[i][j] = random.randint(0, 5)

np.save("init_od_matrix.npy", init_od_matrix)
np.save("zones.npy", zones)

print("Init OD Matrix:")
print(init_od_matrix)

Zones: 5
Init OD Matrix:
[[0 3 5 4 0]
 [1 0 2 4 3]
 [3 0 0 0 0]
 [3 2 1 0 3]
 [2 1 3 5 0]]


In [76]:
# sum of all trrps
total_trips = np.sum(init_od_matrix)
print("Total Trips:", total_trips)

Total Trips: 45


In [77]:
init_od_matrix = np.load("od_matrix.npy")
zones = np.load("zones.npy", allow_pickle=True)

In [78]:
import numpy as np

neshan_od_matrix = np.random.randint(10, 100, size=(n, n))

# Set diagonal to zero
np.fill_diagonal(neshan_od_matrix, 0)
np.save("neshan_od_matrix.npy", neshan_od_matrix)

neshan_od_matrix


array([[ 0, 82, 71, 58, 25],
       [37,  0, 50, 71, 94],
       [80, 38,  0, 89, 36],
       [91, 99, 25,  0, 65],
       [44, 67, 73, 26,  0]], dtype=int32)

In [79]:
import xml.etree.ElementTree as ET
import numpy as np

init_od_matrix = np.load("init_od_matrix.npy")
zones = np.load("zones.npy", allow_pickle=True)

root = ET.Element("data")

interval = ET.SubElement(root, "interval", {
    "begin": "0",
    "end": "3600"
})

for i, zi in enumerate(zones):
    for j, zj in enumerate(zones):

        if i == j:
            continue

        count = init_od_matrix[i][j]
        if count <= 0:
            continue

        ET.SubElement(interval, "tazRelation", {
            "from": str(zi),
            "to": str(zj),
            "count": str(int(count))
        })

ET.ElementTree(root).write("od.xml", encoding="utf-8", xml_declaration=True)

In [80]:
!od2trips -z od.xml -n grids.taz.xml -o trips.xml

Parsing time 0.00
Parsing time 55.65
Parsing time 188.13
Parsing time 345.66
Parsing time 352.32
Parsing time 381.92
Parsing time 410.68
Parsing time 423.68
Parsing time 436.43
Parsing time 448.25
Parsing time 476.94
Parsing time 654.29
Parsing time 674.42
Parsing time 764.28
Parsing time 785.27
Parsing time 842.52
Parsing time 1282.42
Parsing time 1302.21
Parsing time 1600.23
Parsing time 1650.73
Parsing time 1672.50
Parsing time 1720.77
Parsing time 1752.65
Parsing time 1817.33
Parsing time 1822.07
Parsing time 1847.70
Parsing time 1943.63
Parsing time 1997.36
Parsing time 2015.92
Parsing time 2072.59
Parsing time 2092.86
Parsing time 2138.70
Parsing time 2259.21
Parsing time 2279.74
Parsing time 2313.27
Parsing time 2359.25
Parsing time 2488.08
Parsing time 2587.01
Parsing time 2896.69
Parsing time 2942.57
Parsing time 2950.16
Parsing time 3367.17
Parsing time 3451.08
Parsing time 3463.84
Parsing time 3565.79
Success.


In [81]:
!duarouter -n map.net.xml -r trips.xml -o routes.rou.xml --ignore-errors true

Reading up to time step: 55.65
Reading up to time step: 255.65
Reading up to time step: 455.65
Reading up to time step: 655.65
Reading up to time step: 855.65
Reading up to time step: 1055.65
Reading up to time step: 1255.65
Reading up to time step: 1455.65
Reading up to time step: 1655.65
Reading up to time step: 1855.65
Reading up to time step: 2055.65
Reading up to time step: 2255.65
Reading up to time step: 2455.65
Reading up to time step: 2655.65
Reading up to time step: 2855.65
Reading up to time step: 3055.65
Reading up to time step: 3255.65
Reading up to time step: 3455.65
Reading up to time step: 3655.65
Success.


In [83]:
neshan_weights

{'-1035578872': 211.88,
 '-1035578874': 16.13,
 '-1035578887#0': 46.95,
 '-1035578887#1': 80.64,
 '-1035578887#2': 118.61,
 '-1199956439': 4.57,
 '-1385521080': 119.91,
 '-1461800851': 0.2,
 '-164595013': 84.23,
 '-227547757#0': 0.2,
 '-227547757#1': 31.92,
 '-227547757#2': 45.25,
 '-313324028': 54.63,
 '-313324686#0': 113.39,
 '-313324686#1': 159.31,
 '-313324688': 123.08,
 '-39855493#0': 127.27,
 '-39855493#1': 112.99,
 '-39855493#2': 120.22,
 '-39855493#3': 120.08,
 '-39855493#4': 121.43,
 '-39855493#5': 117.75,
 '-42463779#0': 146.77,
 '-42463779#1': 46.8,
 '-42463779#2': 47.36,
 '-42463779#3': 0.2,
 '-4337771#0': 40.65,
 '-4337771#1': 51.36,
 '-4337771#2': 50.7,
 '-4337771#3': 47.27,
 '-4337773': 0.2,
 '-443790550': 182.6,
 '-443790552': 77.32,
 '-443790553': 46.8,
 '-445319881': 27.54,
 '-445321251#0': 121.96,
 '-445321251#1': 20.95,
 '-445321251#2': 17.21,
 '-482796570#0': 199.1,
 '-482796570#1': 66.9,
 '-482796570#2': 117.12,
 '-482796570#3': 120.87,
 '-482796570#4': 120.71,
 '

In [84]:
import xml.etree.ElementTree as ET

file_path = "edges.xml"

tree = ET.parse(file_path)
root = tree.getroot()

edge_travel_time = {}

# SUMO structure: root → interval → edge
for interval in root.findall("interval"):
    for edge in interval.findall("edge"):
        edge_id = edge.get("id")
        traveltime = edge.get("traveltime")

        if edge_id is None or traveltime is None:
            continue

        edge_travel_time[edge_id] = float(traveltime)

print(edge_travel_time)

{'-1035578887#0': 7.21, '-1035578887#1': 11.29, '-1199956439': 1.25, '-164595013': 11.45, '-227547757#1': 4.28, '-227547757#2': 5.67, '-313324028': 7.91, '-39855493#1': 16.04, '-39855493#2': 16.89, '-39855493#5': 16.03, '-42463779#1': 6.53, '-42463779#2': 6.52, '-42463779#3': 0.03, '-4337771#2': 6.94, '-4337771#3': 6.07, '-4337773': 0.06, '-443790553': 7.8, '-445321251#0': 18.0, '-482796570#2': 11.34, '-482796570#3': 11.33, '-482796570#4': 12.69, '-482796570#5': 12.76, '-482796570#6': 12.23, '-488918252': 4.88, '-489251766': 3.82, '-489251768': 2.37, '-489251769#0': 7.35, '-489251769#1': 0.06, '-57388526': 11.4, '-57388567': 10.73, '-57388619#0': 7.46, '-57388619#1': 1.45, '-57388619#2': 2.7, '-57388619#3': 5.98, '-57388627#0': 10.21, '-57388627#1': 2.58, '-57388647#0': 4.2, '-57388666': 3.69, '-57388684': 11.14, '-591969842#0': 9.6, '-591969842#2': 8.35, '-825803618': 2.75, '1035578886': 0.99, '1035578887#0': 6.62, '1035578887#1': 12.24, '1035578888#0': 11.12, '1035578889#0': 3.86, '1

In [86]:
len(neshan_weights),len(init_seed_weights)

(450, 450)

In [89]:
# Symmetric difference (in one but not both)
diff = set(neshan_weights.keys()) ^ set(init_seed_weights.keys())
diff

set()

In [1]:
import nbformat

path = "2.ipynb"

with open(path, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

cells_text = []
for cell in nb.cells:
    if cell.cell_type in ["code", "markdown"]:
        cells_text.append(cell.source)

all_text = "\n\n".join(cells_text)

print(all_text)

import os
import json
import random
import numpy as np
import xml.etree.ElementTree as ET
from sumolib import net

SUMO_HOME = "C:\\Program Files (x86)\\Eclipse\\Sumo"

NET_FILE = "map.net.xml"
TAZ_FILE = "grids.taz.xml"

OD_XML = "od.xml"
TRIPS_FILE = "trips.xml"
ROUTES_FILE = "routes.rou.xml"
EDGES_FILE = "edges.xml"

# !python "C:\Program Files (x86)\Eclipse\Sumo\tools\district\gridDistricts.py" -n map.net.xml -o grids.taz.xml -w 500
# !sumo-gui -c sim.sumocfg

net_obj = net.readNet(NET_FILE)

tree = ET.parse(TAZ_FILE)
root = tree.getroot()

zones = [t.attrib["id"] for t in root.findall("taz")]
n = len(zones)

zone_index = {z: i for i, z in enumerate(zones)}

print("Zones:", n)

neshan_weights = json.load(open("data/neshan-sample-weights.json"))

print("Target edges:", len(neshan_weights))

def save_od_to_xml(od_matrix, filename=OD_XML):
    root = ET.Element("data")
    interval = ET.SubElement(root, "interval", {"begin": "0", "end": "3600"})

    for i, zi in enumerate(zones):
   